# B at D = 128, with and without class weights
The working method (B = `retain_open`: a shadow encoder + decoder keeps what the live encoder must not lose) in the Craftax-shaped grid
world, now with a Craftax-faithful agent and mobs:

* the **agent is an arrow** (its silhouette is the facing, as the sprite is in Craftax), not a white tile with a yellow half;
* the **mob is predictable**: it persists and steps toward the agent with probability 0.7 (else at random), blocked by materials, the border
  and the agent. `twinkle` stays as the only pure noise.

Three runs at D = 128, 100k steps, one seed, the same env. First passes at 15k and 30k (plain and weighted) were not fully settled (at 30k the
weighted run's facing was still rising slowly) and showed the problem the last two runs address: **the weighted run's live encoder loses the
hand (about 0 from step ~10k) while its shadow keeps it (0.9)**. The logs say why: the shadow predictor cannot predict the hand (R² 0.02-0.11
over 30k steps, `ref_rho_hard` 0.03-0.16), so the reference map treats it like noise and releases it; in the plain run the live hand
follows the retention weight down and back up. Waiting does not help (the 100k weighted run was flat at 0.02 from 15k to 32k).

| run | what it answers |
|---|---|
| **B + class weights** | what do ground-truth tile-class weights add on top (agent position, the mob, the arrow)? |
| **B + weights, SIGReg 0.05** | the live rank was 46 of 128 (102 plain): does a 5x stronger SIGReg spread the live encoder and keep the hand? |
| **B + weights + floor on all** | the unconditional version: every shadow direction keeps retention weight >= 0.3 (`ret_floor_all`), whatever its predictability or autocorrelation; nothing represented is released, noise is left to the open target path alone |

The runs go to a **background process**: the notebook only plots, so you can close the tab or restart the kernel and come back. The probe
is an MLP per part at every eval (1000 steps; trained on 12k clips for 1000 steps, closer to the final readout's 20k clips and 2000 steps), on the live encoder (solid) and the shadow (dashed). **Read the shadow first**: an entity
missing from the shadow cannot be recovered by retention; present in the shadow and missing live means retention failed.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import json
import numpy as np
import pandas as pd
from IPython.display import display
from gaf.env_grid import GridCfg, component_masses
from gaf.train import Cfg
from gaf import analysis as A

OUT = "/workspace/results_b128"
SEED, D, STEPS, WORKERS = 0, 128, 100_000, 3   # the runs go the full STEPS (no early stop); 15k and 30k first passes were not fully settled
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=STEPS, basis="excess", floor_steps=1500, floor_warm=True,
              eval_every=1000, eval_parts=12_000, eval_parts_steps=1000, D=D, seed=SEED, env=ENV)
# no "B plain" here, to spare compute: its 100k run from the earlier launch keeps running on the pod (the reference for the late collapse of the
# hand: held until ~50k, lost as the retention weights fall to ~0.1); this list is only what still has to be started
RUNS = {"B + weights": Cfg(arm="retain_open", tag=f"b128-weights-{STEPS // 1000}k", rec_class_power=1.0, **COMMON),
        # the weighted run loses the hand live (shadow keeps it): two attempts, each changing one thing; both log the mechanism diagnostics (gaf/mech.py)
        # every 5000 steps and keep checkpoints, the baseline above does not (its run id would change)
        "B + weights, SIGReg 0.05": Cfg(arm="retain_open", tag=f"b128-weights-lam05-{STEPS // 1000}k", rec_class_power=1.0,
                                        log_mech=5000, ckpt_every=10_000, **{**COMMON, "lam": 0.05}),
        "B + weights + floor on all": Cfg(arm="retain_open", tag=f"b128-weights-floorall-{STEPS // 1000}k", rec_class_power=1.0,
                                          ret_floor_all=True, ret_w_min=0.3, log_mech=5000, ckpt_every=10_000, **COMMON)}
# no sg control: at 15k it held nothing at D = 128 (all parts about -0.1, even the inventory), so D alone does not hold the entities
PARTS = ["hand", "position", "inventory", "facing", "mob", "twinkle"]
pd.set_option("display.precision", 3)

## Masses of the new env (step 0)
Share of the pixel variance of each component. The arrow's facing replaces the marker's (3.0 %); the mob is one tile, as before, but now
predictable.

In [ ]:
m = component_masses(ENV, n=2048)
display(pd.Series({k: f"{v * 100:.2f} %" for k, v in m.items()}, name="share of pixel variance"))

## Launch
Starts the runs in a background process (up to 3 at once on the GPU). **Safe to re-run, and safe while other runs are going:** finished runs
are skipped and so are runs whose `progress.json` was touched in the last 10 minutes (running in another launch), so this only starts what is
new. After killing a launch, wait 10 minutes or delete the folders of the killed runs before relaunching. To stop everything:
`kill -TERM -- -$(cat /workspace/results_b128/launch_*.pid)`. Re-running the cell while they are alive refuses to start twice;
runs that are done are skipped. A run cannot resume: if the pod restarts, the interrupted run starts again.

In [ ]:
A.launch(list(RUNS.values()), OUT, workers=WORKERS)

In [ ]:
display(A.progress(OUT, list(RUNS.values())))
import glob
for f in sorted(glob.glob(f"{OUT}/launch_*.log"))[-2:]:
    print(f)
    print(open(f).read()[-400:])

## Live probes
One panel per part, one colour per run, solid = live encoder, dashed = shadow . Redraws every 30 s and stops
when all runs are done; interrupting the cell stops only the plot.

In [ ]:
A.live_probes(OUT, RUNS, every=30)

## Retention weights
The weight the reference map puts on the shadow directions dominated by each group (`ref_w_<group>`, 1 = fully protected, 0 = free), against the
step. The hand is the `hard` group, position and inventory are `slow`, facing and the mob `fast` (chase mode), twinkle `noise`. A run whose
live hand collapses should show `hard` falling to about 0 here, unless a floor holds it.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True, layout="constrained")
for (name, c), col in zip(RUNS.items(), A.RUN_COLORS):
    try:
        h = pd.read_csv(f"{OUT}/{c.run_id}/metrics.csv")
    except FileNotFoundError:
        continue
    for ax, g in zip(axes, ["hard", "slow", "fast", "noise"]):
        if f"ref_w_{g}" in h:
            ax.plot(h["step"], h[f"ref_w_{g}"], color=col, label=name, marker="o", ms=3)
for ax, g in zip(axes, ["hard", "slow", "fast", "noise"]):
    ax.set(title=f"retention weight, {g} directions", ylim=(-0.05, 1.05))
axes[0].legend(fontsize=7)
plt.show()

## Final readout
Live-encoder MLP probe (R² of each part) and the shadow's, at the end of training, then the change over the last 4000 steps and the swing
inside that window of the live curves (a run that is still moving is provisional).

In [ ]:
df = A.collect(OUT)
rows, shadow, runs = {}, {}, {}
for name, c in RUNS.items():
    r = df[df.run_id == c.run_id]
    if not len(r):
        continue
    rows[name] = {**{p: r[f"pm_{p}"].mean() for p in PARTS}, "rank": r["pm_rank"].mean()}
    if f"sh_pm_{PARTS[0]}" in r and r[f"sh_pm_{PARTS[0]}"].notna().any():
        shadow[name] = {p: r[f"sh_pm_{p}"].mean() for p in PARTS}
    runs[name] = dict(arm=c.arm, tag=c.tag, seed=SEED)  # tag includes the steps
print("live encoder"); display(pd.DataFrame(rows).T.round(3))
print("shadow"); display(pd.DataFrame(shadow).T.round(3))

settle = {}
for name, c in RUNS.items():
    try:
        h = pd.read_csv(f"{OUT}/{c.run_id}/metrics.csv")
    except FileNotFoundError:
        continue
    last = h.step.max()
    W = 4000
    win = h[h.step >= last - W]
    first = h[h.step <= last - W].iloc[-1:] if (h.step <= last - W).any() else h.iloc[:1]
    settle[name] = {f"{p}": (win[f"ev_pm_{p}"].iloc[-1] - first[f"ev_pm_{p}"].iloc[0], win[f"ev_pm_{p}"].max() - win[f"ev_pm_{p}"].min())
                    for p in PARTS if f"ev_pm_{p}" in h}
print("settled? (change, swing) of the live probe over the last 4000 steps; settled = |change| < 0.05 and swing < 0.1")
display(pd.DataFrame(settle).T.map(lambda t: f"{t[0]:+.2f} / {t[1]:.2f}"))

## Reading
* **Shadow first.** An entity that is low in the shadow is not encoded at all (D, weights or the reconstruction are not enough): retention
  cannot help. High in the shadow and low live: the retention/open-target path lost it.
* **The two fixes.** SIGReg 0.05: the live encoder was not spreading (live rank 34-40 of 128 in the weighted run, 104 plain, shadow 120);
  my expectation is that it raises the rank without bringing the hand back, since extra dimensions are filled with whatever is salient.
  Floor on all: retention was releasing hard content. Compare twinkle live too (the floor on all retains noise as well). Check the cost too: inventory and twinkle live (the floor keeps static and
  autocorrelated directions, so live capacity use goes up; twinkle should stay low as its autocorrelation is 0), the live rank, and the
  `ref_ac_n` column of `metrics.csv` (how many directions the floor holds).
* **B vs D alone:** the 15k pass had an `sg` control at D = 128 that held nothing (all parts about -0.1), so what is held here is B's doing.
* **Mob and arrow** are the new content: the mob is predictable now, so unlike before it should be kept, not dropped as noise. `twinkle`
  is the noise entity: B should still drop it (low live).
* Unsettled curves are provisional.